In [3]:
import os
import pandas as pd
import numpy as np
from glob import glob

# ====================================
# CONFIGURATION
# ====================================
BASE_DIR = "FARS"
OUTPUT_PATH = "FARS/cleaned_data/fars_vehicle_cleaned.csv"

VARS_OF_INTEREST = [
    "VSURCOND", "VTRAFCON", "VPAVETYP", "ROUTE", "BODY_TYP", "TOW_VEH",
    "TRLR1GVWR", "TRLR2GVWR", "TRLR3GVWR", "V_CONFIG", "CARGO_BT",
    "TRAV_SP", "M_HARM", "SPEEDREL", "VTRAFWAY", "VNUM_LAN",
    "VSPD_LIM", "VALIGN", "VPROFILE", "VTCONT_F", "P_CRASH1", "P_CRASH2",
    "ST_CASE", "VEH_NO"
]

# ====================================
# LOAD AND CLEAN EACH YEAR
# ====================================
def load_vehicle_file(year_folder):
    """Load vehicle.csv or Vehicle.csv for a given year folder."""
    candidates = glob(os.path.join(year_folder, "[Vv]ehicle.csv"))
    if not candidates:
        print(f" No vehicle file found in {year_folder}")
        return None
    file_path = candidates[0]
    try:
        df = pd.read_csv(file_path, encoding="utf-8", low_memory=False)
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, encoding="latin1", low_memory=False)
    df.columns = df.columns.str.upper().str.strip()
    year = int(os.path.basename(year_folder))
    df["YEAR"] = year
    return df

# ====================================
# HARMONIZATION FUNCTIONS
# ====================================
def harmonize_coded_values(df):
    """Standardize coding and normalize missing values across years."""
    df = df.copy()

    # --- TRAV_SP (Travel Speed) ---
    def clean_trav_sp(row):
        val, year = row.get("TRAV_SP"), row.get("YEAR")
        if pd.isna(val): return np.nan
        try: val = int(val)
        except: return np.nan
        if year <= 2008:
            if val == 0: return 0
            elif 1 <= val <= 96: return val
            elif val == 97: return 152
            elif val in [98, 99]: return np.nan
        elif 2009 <= year <= 2017:
            if val == 0: return 0
            elif 1 <= val <= 151: return val
            elif val == 997: return 152
            elif val in [998, 999]: return np.nan
        else:
            if val == 0: return 0
            elif 1 <= val <= 151: return val
            elif val == 997: return 152
            elif val in [998, 999]: return np.nan
        return np.nan

    if "TRAV_SP" in df.columns:
        df["TRAV_SP"] = df.apply(clean_trav_sp, axis=1)

    # --- SPEEDREL (Speed-Related) ---
    def clean_speedrel(row):
        val = row.get("SPEEDREL")
        if pd.isna(val): return np.nan
        try: val = int(val)
        except: return np.nan
        if val == 0: return 0
        elif val in [1, 2, 3, 4, 5]: return 1
        elif val in [8, 9]: return np.nan
        return np.nan
    if "SPEEDREL" in df.columns:
        df["SPEEDREL"] = df.apply(clean_speedrel, axis=1)

    # --- VSURCOND, VTRAFCON, VPAVETYP ---
    for col, missing_codes in [("VSURCOND",[98,99]), ("VTRAFCON",[97,98,99]), ("VPAVETYP",[8,9])]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df.loc[df[col].isin(missing_codes), col] = np.nan

    # --- M_HARM ---
    if "M_HARM" in df.columns:
        df["M_HARM"] = pd.to_numeric(df["M_HARM"], errors="coerce")
        df.loc[df["M_HARM"].isin([98,99]), "M_HARM"] = np.nan

    # --- VTRAFWAY ---
    if "VTRAFWAY" in df.columns:
        df["VTRAFWAY"] = pd.to_numeric(df["VTRAFWAY"], errors="coerce")
        df.loc[df["VTRAFWAY"].isin([8,9]), "VTRAFWAY"] = np.nan

    # --- VNUM_LAN ---
    if "VNUM_LAN" in df.columns:
        df["VNUM_LAN"] = pd.to_numeric(df["VNUM_LAN"], errors="coerce")
        df.loc[df["VNUM_LAN"].isin([8,9]), "VNUM_LAN"] = np.nan

    # --- VSPD_LIM ---
    if "VSPD_LIM" in df.columns:
        df["VSPD_LIM"] = pd.to_numeric(df["VSPD_LIM"], errors="coerce")
        df.loc[df["VSPD_LIM"].isin([98,99]), "VSPD_LIM"] = np.nan

    # --- VALIGN ---
    if "VALIGN" in df.columns:
        df["VALIGN"] = pd.to_numeric(df["VALIGN"], errors="coerce")
        df.loc[df["VALIGN"].isin([8,9]), "VALIGN"] = np.nan

    # --- VPROFILE ---
    if "VPROFILE" in df.columns:
        df["VPROFILE"] = pd.to_numeric(df["VPROFILE"], errors="coerce")
        df.loc[df["VPROFILE"].isin([8,9]), "VPROFILE"] = np.nan

    # --- VTCONT_F ---
    if "VTCONT_F" in df.columns:
        df["VTCONT_F"] = pd.to_numeric(df["VTCONT_F"], errors="coerce")
        df.loc[df["VTCONT_F"].isin([8,9]), "VTCONT_F"] = np.nan

    # --- P_CRASH1 & P_CRASH2 ---
    for col in ["P_CRASH1", "P_CRASH2"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df.loc[df[col].isin([98,99]), col] = np.nan

    # --- BODY_TYP ---
    if "BODY_TYP" in df.columns:
        df["BODY_TYP"] = pd.to_numeric(df["BODY_TYP"], errors="coerce")
        df.loc[df["BODY_TYP"].isin([97,98,99]), "BODY_TYP"] = np.nan

    # --- ROUTE ---
    if "ROUTE" in df.columns:
        df["ROUTE"] = pd.to_numeric(df["ROUTE"], errors="coerce")
        df.loc[df["ROUTE"].isin([97,98,99]), "ROUTE"] = np.nan

    # --- TOW_VEH ---
    if "TOW_VEH" in df.columns:
        df["TOW_VEH"] = pd.to_numeric(df["TOW_VEH"], errors="coerce")
        df.loc[df["TOW_VEH"].isin([7,8,9]), "TOW_VEH"] = np.nan

    # --- TRLR1/2/3GVWR ---
    for col in ["TRLR1GVWR", "TRLR2GVWR", "TRLR3GVWR"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df.loc[df[col].isin([98,99]), col] = np.nan

    # --- V_CONFIG ---
    if "V_CONFIG" in df.columns:
        df["V_CONFIG"] = pd.to_numeric(df["V_CONFIG"], errors="coerce")
        df.loc[df["V_CONFIG"].isin([98,99]), "V_CONFIG"] = np.nan

    # --- CARGO_BT ---
    if "CARGO_BT" in df.columns:
        df["CARGO_BT"] = pd.to_numeric(df["CARGO_BT"], errors="coerce")
        df.loc[df["CARGO_BT"].isin([98,99]), "CARGO_BT"] = np.nan

    return df

# ====================================
# COMBINE ALL YEARS
# ====================================
def combine_vehicle_data(base_dir):
    year_folders = sorted([f.path for f in os.scandir(base_dir) if f.is_dir()])
    all_dfs = []
    for folder in year_folders:
        df = load_vehicle_file(folder)
        if df is None:
            continue
        cols = [c for c in VARS_OF_INTEREST if c in df.columns]
        df = df[cols + ["YEAR"]]
        df = harmonize_coded_values(df)

        # Create unique ID (Year-Case)
        if "ST_CASE" in df.columns:
            df["ID"] = "FARS_" + df["YEAR"].astype(str) + "_" + df["ST_CASE"].astype(str)
        else:
            df["ID"] = "FARS_" + df["YEAR"].astype(str) + "_MISSING"

        all_dfs.append(df)
        print(f" Processed {os.path.basename(folder)}: {df.shape[0]} rows, {df.shape[1]} cols")

    return pd.concat(all_dfs, ignore_index=True)

# ====================================
# DATA QUALITY CHECK
# ====================================
def summarize_data_quality(df):
    print("\n=== DATA QUALITY SUMMARY ===")
    missing_pct = df.isna().mean() * 100
    print("Missingness (%):")
    print(missing_pct.sort_values(ascending=False))
    print("\nRanges / Unique Values:")
    for col in df.columns:
        if df[col].dtype in [np.int64, np.float64]:
            print(f"{col}: min={df[col].min()}, max={df[col].max()}")
        else:
            print(f"{col}: {df[col].nunique()} unique values")

# ====================================
# MAIN SCRIPT
# ====================================
if __name__ == "__main__":
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    fars_clean = combine_vehicle_data(BASE_DIR)
    summarize_data_quality(fars_clean)

    if {"ID", "VEH_NO"}.issubset(fars_clean.columns):
        dupes = fars_clean.duplicated(subset=["ID", "VEH_NO"], keep=False)
        dup_df = fars_clean.loc[dupes, ["ID", "VEH_NO"]]
        n_dupes = dup_df.shape[0]
        print(f"\n Duplicate check: Found {n_dupes} duplicate vehicle-case entries")
        if n_dupes > 0:
            print(dup_df.head())

    fars_clean.to_csv(OUTPUT_PATH, index=False)
    print(f"\n Cleaned data saved to: {OUTPUT_PATH}")


 Processed 2016: 52714 rows, 22 cols
 Processed 2017: 53128 rows, 22 cols
 Processed 2018: 52286 rows, 22 cols
 Processed 2019: 51623 rows, 22 cols
 Processed 2020: 54552 rows, 25 cols
 Processed 2021: 61802 rows, 25 cols
 Processed 2022: 60765 rows, 25 cols
 Processed 2023: 58319 rows, 25 cols
 No vehicle file found in FARS\cleaned_data

=== DATA QUALITY SUMMARY ===
Missingness (%):
TRAV_SP      60.701859
TRLR1GVWR    49.660931
TRLR2GVWR    47.212532
TRLR3GVWR    47.121784
VPAVETYP     26.110933
VTRAFCON      8.846804
VPROFILE      8.700350
VTCONT_F      8.522897
SPEEDREL      5.510244
VSPD_LIM      3.501434
P_CRASH2      2.943469
VALIGN        2.119774
BODY_TYP      2.066538
CARGO_BT      2.009259
V_CONFIG      1.555294
VSURCOND      1.519804
P_CRASH1      1.500936
VNUM_LAN      1.116380
VTRAFWAY      0.576834
TOW_VEH       0.355130
M_HARM        0.112312
ST_CASE       0.000000
VEH_NO        0.000000
YEAR          0.000000
ID            0.000000
dtype: float64

Ranges / Unique Values